```markdown
# Colab Pro 環境設定

Google Colab Proをご利用とのことですので、GPUの利用状況確認とシステムメモリの情報を確認するコードを記述します。

## GPU設定と確認
Colab Proでは高性能なGPUが利用可能です。以下のコードでGPUが利用可能かを確認できます。TensorFlowやPyTorchなどのライブラリを使用する際に、GPUが正しく認識されているかを確認することは重要です。
```

In [21]:
import tensorflow as tf

print("GPU が利用可能か:", tf.config.list_physical_devices('GPU'))

# もし GPU が利用可能な場合、詳細情報を表示
!nvidia-smi

GPU が利用可能か: []
/bin/bash: line 1: nvidia-smi: command not found


```markdown
## システムメモリの確認
Colab Proではより多くのメモリが提供されます。以下のコードで現在のシステムメモリの使用状況と合計メモリ量を確認できます。
```

In [22]:
import psutil

# メモリ情報を取得
mem = psutil.virtual_memory()
total_memory_gb = mem.total / (1024**3) # バイトをギガバイトに変換
available_memory_gb = mem.available / (1024**3)
used_memory_gb = mem.used / (1024**3)

print(f"総メモリ: {total_memory_gb:.2f} GB")
print(f"利用可能メモリ: {available_memory_gb:.2f} GB")
print(f"使用済みメモリ: {used_memory_gb:.2f} GB")

# より詳細なメモリ情報 (linux コマンド)
!cat /proc/meminfo

総メモリ: 12.67 GB
利用可能メモリ: 11.04 GB
使用済みメモリ: 1.35 GB
MemTotal:       13286944 kB
MemFree:         7966820 kB
MemAvailable:   11573196 kB
Buffers:          156104 kB
Cached:          3642400 kB
SwapCached:            0 kB
Active:          1141076 kB
Inactive:        3945044 kB
Active(anon):       1448 kB
Inactive(anon):  1288352 kB
Active(file):    1139628 kB
Inactive(file):  2656692 kB
Unevictable:          20 kB
Mlocked:              20 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:              1212 kB
Writeback:             0 kB
AnonPages:       1285160 kB
Mapped:          1029004 kB
Shmem:              2172 kB
KReclaimable:     102212 kB
Slab:             142300 kB
SReclaimable:     102212 kB
SUnreclaim:        40088 kB
KernelStack:        6272 kB
PageTables:        12112 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     6643472 kB
Committed_AS:    4296428 kB
VmallocTotal:   3435973836

In [23]:
import sys

from google.colab import drive
drive.mount('/content/drive')

# プロジェクトルートの設定（Google Drive）
PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026") # Google Drive内の正しいパスに修正
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 03_baseline_lgbm
LightGBM単体モデルのベースライン実験

In [24]:
import datetime
import os
from pathlib import Path

import numpy as np
import pandas as pd

# モジュールのインポート
from common.lgbm.lgbm_model import run_lgb
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

# 乱数シードの固定
SEED = 42
seed_everything(seed=SEED)

# ターゲット列とID列の設定
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

In [25]:
# スクリプト名・日付・保存パスの設定
SCRIPT_NAME = "04_baseline_lgbm_for_google_colab"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# ログディレクトリ
LOG_DIR = PROJECT_ROOT / "logs"

logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

# 出力ディレクトリ
OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

[2026-08-05 14:22:46] [INFO] === [04_baseline_lgbm_for_google_colab] 実験開始 ===


INFO:04_baseline_lgbm_for_google_colab:=== [04_baseline_lgbm_for_google_colab] 実験開始 ===


In [26]:
# データの読み込み
INPUT_DIR = PROJECT_ROOT / "data" / "input"

# 属性データの読み込み (社員1名 = 1行)
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")

# 月次データの読み込み (社員1名 × 24か月 = 複数行)
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

[2026-08-05 14:22:47] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:04_baseline_lgbm_for_google_colab:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-05 14:22:47] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:04_baseline_lgbm_for_google_colab:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


In [27]:
def transform_monthly_to_wide_all(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """月次データ(0〜23か月)の全カラムを横持ち(ピボット)にし、社員1名=1行に変換する"""
    val_cols = [
        col for col in monthly_df.columns if col not in ["社員ID", "経過月数"]
    ]

    monthly_wide = monthly_df.pivot(
        index="社員ID", columns="経過月数", values=val_cols
    )

    monthly_wide.columns = [
        f"{col}_m{month}" for col, month in monthly_wide.columns
    ]
    monthly_wide = monthly_wide.reset_index()

    return monthly_wide


# 月次データを横持ち化
train_monthly_wide = transform_monthly_to_wide_all(train_monthly)
test_monthly_wide = transform_monthly_to_wide_all(test_monthly)

# 属性データと月次横持ちデータを結合
train_df = pd.merge(train_persona, train_monthly_wide, on="社員ID", how="left")
test_df = pd.merge(test_persona, test_monthly_wide, on="社員ID", how="left")

logger.info(f"結合後 Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")

[2026-08-05 14:22:48] [INFO] 結合後 Train Shape: (2761, 668), Test Shape: (2502, 667)


INFO:04_baseline_lgbm_for_google_colab:結合後 Train Shape: (2761, 668), Test Shape: (2502, 667)


In [28]:
def build_features(
    train: pd.DataFrame, test: pd.DataFrame, target_col: str, id_col: str
):
    """特徴量エンジニアリングを行う関数"""
    train_proc = train.copy()
    test_proc = test.copy()

    # 非数値列（文字列・日付列）を除外
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()

    if id_col in non_num_cols:
        non_num_cols.remove(id_col)

    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")

    # 特徴量とターゲットに分割
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")

    input_data = {"X_train": X_train, "y_train": y_train, "X_test": X_test}

    return input_data, test_proc[id_col]


# 特徴量エンジニアリング実行
input_data, test_ids = build_features(
    train_df, test_df, target_col=TARGET_COL, id_col=ID_COL
)

logger.info(f"X_train Shape: {input_data['X_train'].shape}")
logger.info(f"X_test Shape: {input_data['X_test'].shape}")

[2026-08-05 14:22:49] [INFO] X_train Shape: (2761, 3)


INFO:04_baseline_lgbm_for_google_colab:X_train Shape: (2761, 3)


[2026-08-05 14:22:49] [INFO] X_test Shape: (2502, 3)


INFO:04_baseline_lgbm_for_google_colab:X_test Shape: (2502, 3)


In [29]:
# LightGBMパラメータ
lgb_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
    "objective": "binary",
    "metric": "binary_logloss",
    "early_stopping_rounds": 50,
    "verbose": -1,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "n_estimators": 1000,
}

logger.info("LightGBMパラメータ:")
for key, value in lgb_params.items():
    logger.info(f"  {key}: {value}")

[2026-08-05 14:22:49] [INFO] LightGBMパラメータ:


INFO:04_baseline_lgbm_for_google_colab:LightGBMパラメータ:


[2026-08-05 14:22:49] [INFO]   n_splits: 5


INFO:04_baseline_lgbm_for_google_colab:  n_splits: 5


[2026-08-05 14:22:49] [INFO]   seed: 42


...

INFO:04_baseline_lgbm_for_google_colab:  num_leaves: 31


[2026-08-05 14:22:49] [INFO]   learning_rate: 0.05


INFO:04_baseline_lgbm_for_google_colab:  learning_rate: 0.05


[2026-08-05 14:22:49] [INFO]   n_estimators: 1000


INFO:04_baseline_lgbm_for_google_colab:  n_estimators: 1000


In [30]:
logger.info("--- LightGBM トレーニング開始 ---")
lgb_res, _ = run_lgb(data=input_data, params=lgb_params)

# CV スコア計算
lgb_cv = calculate_logloss(input_data["y_train"], lgb_res["oof_preds"])
logger.info(f"LightGBM CV Score (LogLoss): {lgb_cv:.4f}")

[2026-08-05 14:22:49] [INFO] --- LightGBM トレーニング開始 ---


INFO:04_baseline_lgbm_for_google_colab:--- LightGBM トレーニング開始 ---


[2026-08-05 14:22:49] [INFO] LightGBM CV Score (LogLoss): 0.6698


INFO:04_baseline_lgbm_for_google_colab:LightGBM CV Score (LogLoss): 0.6698


In [31]:
# 提出ファイル作成
sub = pd.DataFrame({ID_COL: test_ids, TARGET_COL: lgb_res["test_preds"]})

sub.to_csv(SUBMISSION_PATH, index=False,header=False)
logger.info(f"提出ファイルを保存しました: {SUBMISSION_PATH}")
logger.info("=== 実験完了 ===")

print(f"\n提出ファイル: {SUBMISSION_PATH}")
print(f"CV Score (LogLoss): {lgb_cv:.4f}")

[2026-08-05 14:22:50] [INFO] 提出ファイルを保存しました: /content/drive/MyDrive/jaggle_2026/data/output/20260805/20260805_04_baseline_lgbm_for_google_colab.csv


INFO:04_baseline_lgbm_for_google_colab:提出ファイルを保存しました: /content/drive/MyDrive/jaggle_2026/data/output/20260805/20260805_04_baseline_lgbm_for_google_colab.csv


[2026-08-05 14:22:50] [INFO] === 実験完了 ===


INFO:04_baseline_lgbm_for_google_colab:=== 実験完了 ===



提出ファイル: /content/drive/MyDrive/jaggle_2026/data/output/20260805/20260805_04_baseline_lgbm_for_google_colab.csv
CV Score (LogLoss): 0.6698
